In [82]:
import numpy as np

In [83]:
# Funções Sigmoid e ReLU para ativação e suas derivadas

def sigmoid(x):
    return 1 / (1+np.exp(-x))

def ReLU(x):
    return np.maximum(0,x)

def ReLU_deriv(x):
    return (x > 0).astype(float)
    
def sigmoid_deriv(x):
    return sigmoid(x) * (1-sigmoid(x))


$$Z = Wx + b$$

$$
\begin{bmatrix}
W_{1,1} & W_{1,2} \\
W_{2,1} & W_{2,2} \\
W_{3,1} & W_{3,2} \\
W_{4,1} & W_{4,2}
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2
\end{bmatrix}
+
\begin{bmatrix}
b_1 \\
b_2 \\
b_3 \\
b_4
\end{bmatrix}
=
\begin{bmatrix}
Z_1 \\
Z_2 \\
Z_3 \\
Z_4
\end{bmatrix}
$$


Podemos verificar que o numero de linhas da matriz de pesos $W$ no mostra a quantidade de neurônios na rede. O número de colunas é igual ao número de entradas, permitindo a multiplicação matricial. Ou seja, se temos $n$ entradas e queremos $p$ neurônios, devemos ter uma matriz $W$ com shape $p \times n$. E a matriz de bias $b$ deve ter valor igual ao número $p$ de neurônios.

Obs: Em numpy, mantemos os valores em vetores linha, fazendo a transposta das matrizes demonstradas matemáticamente, assim economizando recursos computacionais, porem devemos estar atendos para realizar as operações de multiplicação na ordem certa.




Ainda na primeira camada, "jogamos" o vetor $Z$ na função de ativação ReLU, que "destrói" qualquer resultado negativo e mantém os valores positivos:

$$f(x) = max(0,x)$$

$$A_1 = ReLU(Z)$$

A função ReLU quebra a linearidade da rede, permitindo o aprendizado de curvas e formas mais complexas. Também, ao zerar valores negativos, a função ReLU faz com que o modelo foque em valores que realmente importam no resultado.

In [84]:
#PRIMEIRA CAMADA

entrada = np.array([0,1])
Y = 0.666 # Resultado esperado

W = np.random.randn(2,4)
b = np.zeros(4)
Z1 = np.dot(entrada, W) + b
A1 = ReLU(Z1)

print(f"Entrada: {entrada}")
print(f"Entrada Shape: {entrada.shape}")
print(30*'-')
print(f"Pesos: {W}")
print(f"Pesos Shape: {W.shape}")
print(30*'-')
print(f"Bias: {b}")
print(f"Bias Shape: {b.shape}")
print(30*'-')
print(f"Z: {Z}")
print(f"Z Shape: {Z.shape}")
print(30*'-')
print(f"A1: {A1}")
print(f"A1 Shape: {A1.shape}")
print(30*'-')


Entrada: [0 1]
Entrada Shape: (2,)
------------------------------
Pesos: [[-0.07754924  0.30360685  0.52220564  0.99016136]
 [-0.14396624 -0.60069075  0.39919557 -0.09447065]]
Pesos Shape: (2, 4)
------------------------------
Bias: [0. 0. 0. 0.]
Bias Shape: (4,)
------------------------------
Z: [-0.19223006 -0.1923155   0.07252851  0.55059092]
Z Shape: (4,)
------------------------------
A1: [0.         0.         0.39919557 0.        ]
A1 Shape: (4,)
------------------------------


A saída da primeira camada, $A_1$, agora será a entrada na segunda camada, que fará o mesmo procedimento algébrico, multiplicando $A_1$ por uma Matriz $W_2$ de pesos e somar ao vetor $b_2$ de bias.
$$ Z_2 = A_1 W_2 + b_2 $$

Assim como na primeira camada vamos aplicar a função de ativação no vetor $Z_2$. Dessa vez nossa função de ativação será a Sigmoid, que permite que a saída esteja apenas entre 0 e 1
$$ \sigma(x) = \frac{1} { 1 + e^{-x}} $$

Logo:

$$ A_2 = \sigma(Z_2)$$

Para obtermos uma saída da segunda camada, que nesse caso é nossa camada de saída, precisamos que a matriz $W_2$ seja $4 \times 1$ para "transformar os 4 valores do vetor $A_1$ em apenas um escalar, que será somado com $b_2$ com shape $1 \times 1$ e depois aplicado em $\sigma(x)$.

In [85]:
#SEGUNDA CAMADA

W2 = np.random.randn(4,1)
b2 = np.zeros(1)
Z2 = np.dot(A1,W2) + b2
A2 = sigmoid(Z2) # Saída da camada dois

print(f"Pesos: {W2}")
print(f"Pesos Shape: {W2.shape}")
print(30*'-')
print(f"Bias: {b2}")
print(f"Bias Shape: {b2.shape}")
print(30*'-')
print(f"A2: {A2}")
print(f"A2 Shape: {A2.shape}")
print(30*'-')


Pesos: [[ 0.99561255]
 [-0.23597191]
 [-0.26772147]
 [-0.44255808]]
Pesos Shape: (4, 1)
------------------------------
Bias: [0.]
Bias Shape: (1,)
------------------------------
A2: [0.4733071]
A2 Shape: (1,)
------------------------------


Agora que temos a saída da rede, precisamos comparar ela com a saída esperada através da Função de Perda. Nesse caso, estamos usando o Erro Quadrático Médio (MSE), onde: $$Loss = (A_2 - Y)^2$$
Para que possamos propagar o erro atraves da rede, para encontrar os "culpados" fazemos o processo de back propagation, usando os gradientes.

Primeiramente, precisamos saber quanto o erro varia de acordo com $Z_2$. Como Loss depende de $A_2$, que depende de $Z_2$ através da função sigmoid, precisamos usar a regra da Cadeia:
$$ \frac{\partial Loss}{\partial Z_2} = \frac{\partial Loss}{\partial A_2} \cdot \frac{\partial A_2}{\partial Z_2} $$
Derivada da função $Loss$: $$ \frac{\partial}{\partial A_2}(A_2 - Y)^2 = 2(A_2 - Y)$$
Derivada da função $Sigmoid$: $$ \sigma'(Z_2) = \sigma(Z_2)(1 - \sigma(Z_2))$$
Logo: $$\frac{\partial Loss}{\partial Z_2} = 2(A_2 - Y) \cdot \sigma'(Z_2)$$


In [86]:
loss = (A2 - Y)**2
dZ2 = 2 * (A2 - Y) * sigmoid_deriv(Z2)
print(f"Loss: {loss}")
print(f"dZ2: {dZ2}")


Loss: [0.03713056]
dZ2: [-0.09607186]


Agora que temos o erro na saída, precisamos calcular o quanto de "culpa" tem os pesos na camada dois, $W_2$. 
$$\frac {\partial Loss}{\partial W_2} = \frac{\partial Loss}{\partial Z_2} \cdot \frac{\partial Z_2}{\partial W_2}$$
Ja calculamos a relação de dependencia entre $Loss$ e $Z_2$ antes, agora precisamos de $\frac{\partial Z_2}{\partial W_2}$. Como $Z_2 = A_1 \cdot W_2 + b_2$, a derivada de $Z_2$ em $W_2$ é $A_1$.
Logo: $$\frac{\partial Loss}{\partial W_2} = \frac{\partial Loss}{\partial Z_2} \cdot \frac{\partial Z_2}{\partial W_2} = dZ_2 \cdot A_1$$


Obs: Inicialmente estavamos trabalhando linhas no codigo, ao invés de colunas. Mas no back propagation usamos um reshape em $dW_2$ e $A_1$ para garantir que ocorra produto matricial, e não uma operação escalar, e também para que retorne um vetor $4 \times 1$ que "encaixa" na matriz $W_2$

In [87]:
dW2 = np.dot(A1.reshape(4,1), dZ2.reshape(1,1))
db2 = dZ2 * 1
print(f"dW2: {dW2}")
print(f"db2: {db2}")

dW2: [[ 0.        ]
 [ 0.        ]
 [-0.03835146]
 [ 0.        ]]
db2: [-0.09607186]


Para retornarmos à camada oculta, precisamos calcular a variação de $Loss$ em relação a $Z_1$. Como $Z_1$ depende de $A_1$, precisamos primeiro calcular a variação de $Loss$ em $A_1$. Pra isso podemos usar a derivada parcial de $Loss$ em $Z_2$, que já calculamos, pois $Z_2$ depende de $A_1$:
$$\frac{\partial Loss}{\partial A_1} =   \frac{\partial Loss}{\partial Z_2} \cdot \frac{\partial Z_2}{\partial A_1}$$
Sabemos que $\frac{\partial Loss}{\partial Z_2} = 2(A_2 - Y) \cdot \sigma'(Z_2)$.
Podemos calcular tambem $\frac{\partial Z_2}{\partial A_1}$ e chegamos em $W_2$
Logo: $$\frac{\partial Loss}{\partial A_1} = 2(A_2 - Y) \cdot \sigma'(Z_2) \cdot W_2$$



Agora que temos o quanto $Loss$ varia em relação a $A_1$, podemos calcular a varição de $Loss$ em $Z_1$:
$$\frac{\partial Loss}{\partial Z_1} =   \frac{\partial Loss}{\partial A_1} \cdot \frac{\partial A_1}{\partial Z_1}$$

A relação entre $A_1$ e $Z_1$ é a função $ReLU(Z_1)$. Portando, para saber a relação de variação, fazemos a derivada parcial da função em $Z_1$:
$$\frac{\partial A_1}{\partial Z_1} = ReLU'(Z_1)$$

Concluimos então: 
$$\frac{\partial Loss}{\partial Z_1} = (\frac{\partial Loss}{\partial Z_2} \cdot W_2^T) \odot ReLU'(Z_1)$$

No Forward Pass, $W_2$ transforma 4 valores em 1 saída. No Back Propagation Usamos a transposta $W^2T$ pois o erro $dZ_2$ tem tamanho 1 e a camada oculta tem 4 neurônios então queremos o inverso do Forward, tranformar 1 valor em 4. 





In [88]:
dZ1 = (np.dot(dZ2, W2.T) * ReLU_deriv(Z1))
dW1 = np.dot(entrada.reshape(2,1), dZ1.reshape(1,4))
db1 = dZ1

Agora que possuimos todos os gradientes necessários para calcular os erros, podemos montar o loop de treinamento. Mas antes, precisamos escolher um valor de learning_rate que é o quanto o modelo "caminha" até o erro zero. O valor deve ser razoável, de forma que não demore exageradamente para chegar no erro zero (learning_rate muito pequeno) e nem "pule" através do erro zero (learning_rate muito grande).

In [101]:

learning_rate = 0.1
epochs = 10000 
Y = np.array([
    [0],[1],[1],[0]
])
entrada = np.array([
    [0,0],
    [0,1],
    [1,0],
    [1,1]
    ])


W1 = np.random.randn(2,4)
b1 = np.zeros((1,4))
W2 = np.random.randn(4,1)
b2 = np.zeros((1,1))

for epoch in range(epochs):
    # FORWARD PASS 
    Z1 = np.dot(entrada, W1) + b1
    A1 = ReLU(Z1)
    
    Z2 = np.dot(A1, W2) + b2
    A2 = sigmoid(Z2)

    # LOSS 
    loss = np.mean((A2 - Y)**2)

    # BACKPROPAGATION
    # Camada de Saída
    dZ2 = 2 * (A2 - Y) * sigmoid_deriv(Z2)
    dW2 = np.dot(A1.T, dZ2)
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    # Camada Oculta
    dZ1 = np.dot(dZ2, W2.T) * ReLU_deriv(Z1) # Usamos Z1 aqui
    dW1 = np.dot(entrada.T, dZ1)
    db1 = np.sum(dZ1, axis=0, keepdims=True)
    
    #  ATUALIZAÇÃO
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    
    # Monitoramento
    if epoch % 1000 == 0:
        print(f"Época {epoch} - Loss: {loss:.6f}")

print("\n--- Tabela de Resultados ---")
print("Entrada [0,0] | Alvo: 0 | Previsão:", A2[0][0].round(4))
print("Entrada [0,1] | Alvo: 1 | Previsão:", A2[1][0].round(4))
print("Entrada [1,0] | Alvo: 1 | Previsão:", A2[2][0].round(4))
print("Entrada [1,1] | Alvo: 0 | Previsão:", A2[3][0].round(4))

Época 0 - Loss: 0.367742
Época 1000 - Loss: 0.003030
Época 2000 - Loss: 0.001079
Época 3000 - Loss: 0.000632
Época 4000 - Loss: 0.000440
Época 5000 - Loss: 0.000335
Época 6000 - Loss: 0.000270
Época 7000 - Loss: 0.000225
Época 8000 - Loss: 0.000193
Época 9000 - Loss: 0.000168

--- Tabela de Resultados ---
Entrada [0,0] | Alvo: 0 | Previsão: 0.0093
Entrada [0,1] | Alvo: 1 | Previsão: 0.9801
Entrada [1,0] | Alvo: 1 | Previsão: 0.9932
Entrada [1,1] | Alvo: 0 | Previsão: 0.0081
